In [7]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=SBCoFVnfzo1qAX3JklPq5i4gJIWJe7&access_type=offline&code_challenge=uHpYaXY7OEvLge3NcX97aJic7bfIq61TiFhtfNqBeOs&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [9]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=m3UDMbvUt5BcGTpsDdYYwgUIwpkNLo&access_type=offline&code_challenge=jVehlml18X1cYrzJharmWWR5BPlKfO98EbS_-5hgFO8&code_challenge_method=S256


You are now logged in as [yt4@sanger.ac.uk].
Your current project is [open-targets-genetics-dev].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [3]:
!gcloud config set project open-targets-genetics-dev

Updated property [core/project].


In [1]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep

Loading BokehJS ...

In [2]:
hail_dir = os.path.dirname(hl.__file__)
session = Session(hail_home=hail_dir, start_hail=True, extended_spark_conf={
    "spark.driver.memory": "12g","spark.kryoserializer.buffer.max": "500m","spark.driver.maxResultSize":"2g",
    'spark.hadoop.fs.gs.requester.pays.buckets': 'requester-pays-bucket1,requester-pays-bucket2',
    'spark.hadoop.fs.gs.requester.pays.project.id': 'open-targets-genetics-dev',
    "spark.hadoop.fs.gs.requester.pays.mode":"AUTO"})

24/10/16 11:53:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.3.4
SparkUI available at http://mib118093s.internal.sanger.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.127-bb535cd096c5
LOGGING: writing to /dev/null


In [3]:
from __future__ import annotations

from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType

from gentropy.common.session import Session
from gentropy.datasource.gwas_catalog.study_index import StudyIndexGWASCatalogParser
from gentropy.datasource.gwas_catalog.study_index_ot_curation import (
    StudyIndexGWASCatalogOTCuration,
)

In [4]:
catalog_study_files=["gs://gwas_catalog_inputs/gwas_catalog_download_studies.tsv"]
catalog_ancestry_files=["gs://gwas_catalog_inputs/gwas_catalog_download_ancestries.tsv"]
gwas_catalog_study_curation_file="gs://genetics-portal-dev-analysis/yt4/gwas_catalog_curation/20241004_output_curation.tsv"
sumstats_qc_path="gs://genetics-portal-dev-analysis/yt4/20241015_GwasCatQCLogs"
harmonised_studies_index_path=None

In [5]:
# Core Study Index Generation:
study_index = StudyIndexGWASCatalogParser.from_source(
    session.spark.read.csv(list(catalog_study_files), sep="\t", header=True),
    session.spark.read.csv(list(catalog_ancestry_files), sep="\t", header=True),
)

# Annotate with curation if provided:
if gwas_catalog_study_curation_file:
    if gwas_catalog_study_curation_file.endswith(
        ".tsv"
    ) | gwas_catalog_study_curation_file.endswith(".tsv"):
        gwas_catalog_study_curation = StudyIndexGWASCatalogOTCuration.from_csv(
            session, gwas_catalog_study_curation_file
        )
    elif gwas_catalog_study_curation_file.startswith("http"):
        gwas_catalog_study_curation = StudyIndexGWASCatalogOTCuration.from_url(
            session, gwas_catalog_study_curation_file
        )
    else:
        raise ValueError(
            "Only CSV/TSV files or URLs are accepted as curation file."
        )
    study_index = study_index.annotate_from_study_curation(
        gwas_catalog_study_curation
    )

# Annotate with has summary statistics if provided:
if harmonised_studies_index_path:
    harmonised_studies = session.spark.read.csv(
        harmonised_studies_index_path, header=False
    )
    study_index = study_index.annotate_sumstats_info(harmonised_studies)

# Annotate with sumstats QC if provided:
if sumstats_qc_path:
    schema = StructType(
        [
            StructField("studyId", StringType(), True),
            StructField("mean_beta", DoubleType(), True),
            StructField("mean_diff_pz", DoubleType(), True),
            StructField("se_diff_pz", DoubleType(), True),
            StructField("gc_lambda", DoubleType(), True),
            StructField("n_variants", LongType(), True),
            StructField("n_variants_sig", LongType(), True),
        ]
    )
    sumstats_qc = session.spark.read.schema(schema).parquet(
        sumstats_qc_path, recursiveFileLookup=True
    )
    study_index = study_index.annotate_sumstats_qc(sumstats_qc)


In [12]:
study_index.df.show(2,truncate=False)

+----------+----------------------+------------------------+------+----------------------------------------------------------------+---------+---------+------------------+----------------------------------------------------------------------------------------------------+--------+--------------------------+-------+----------------------------------+-------------+--------+---------------------+-------------------+---------+---------------+---------------+---------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+-----------+
|studyId   |publicationFirstAuthor|traitFromSourceMappedIds|nCases|initialSampleSize                                               |nControls|projectId|publicationJournal|publicationTitle                                                                                    |nSamples|traitFromSource           |cohorts|b

In [23]:
df=study_index.df
df.count()

52441

In [26]:
df = df.join(
    sumstats_qc, how="left", on="studyId"
) 

In [28]:
df = df.drop(
    "mean_beta",
    "mean_diff_pz",
    "se_diff_pz",
    "gc_lambda",
    "n_variants",
    "n_variants_sig",
)

In [29]:
df.printSchema()

root
 |-- studyId: string (nullable = false)
 |-- publicationFirstAuthor: string (nullable = true)
 |-- traitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- nCases: integer (nullable = true)
 |-- initialSampleSize: string (nullable = true)
 |-- nControls: integer (nullable = true)
 |-- projectId: string (nullable = false)
 |-- publicationJournal: string (nullable = true)
 |-- publicationTitle: string (nullable = true)
 |-- nSamples: integer (nullable = true)
 |-- traitFromSource: string (nullable = false)
 |-- cohorts: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- backgroundTraitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- analysisFlags: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- pubmedId: string (nullable = true)
 |-- ldPopulationStructure: array (nullable = true)
 |    |-- element: struct (containsNull = false)
 |   

In [ ]:
df_exploded = study_index.df.withColumn("analysisFlags_e", f.explode("analysisFlags"))

# Group by the exploded elements and count the occurrences
df_counts = df_exploded.groupBy("analysisFlags_e").count()
df_counts.show(truncate=False)

In [11]:
# Group by the exploded elements and count the occurrences
df_counts = study_index.df.groupBy("studyType").count()
df_counts.show(truncate=False)

+----------+-----+
|studyType |count|
+----------+-----+
|gwas      |34140|
|pQTL      |16736|
|Microbiome|1565 |
+----------+-----+



In [7]:
study_index.df.printSchema()

root
 |-- studyId: string (nullable = false)
 |-- publicationFirstAuthor: string (nullable = true)
 |-- traitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- nCases: integer (nullable = true)
 |-- initialSampleSize: string (nullable = true)
 |-- nControls: integer (nullable = true)
 |-- projectId: string (nullable = false)
 |-- publicationJournal: string (nullable = true)
 |-- publicationTitle: string (nullable = true)
 |-- nSamples: integer (nullable = true)
 |-- traitFromSource: string (nullable = false)
 |-- cohorts: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- backgroundTraitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- analysisFlags: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- pubmedId: string (nullable = true)
 |-- ldPopulationStructure: array (nullable = true)
 |    |-- element: struct (containsNull = false)
 |   

In [7]:
study_index.df.filter(f.col("sumStatQCPerformed")==True).count()

52441

In [5]:
session.spark.read.csv(catalog_study_files, sep="\t", header=True)

DataFrame[DATE ADDED TO CATALOG: string, PUBMED ID: string, FIRST AUTHOR: string, DATE: string, JOURNAL: string, LINK: string, STUDY: string, DISEASE/TRAIT: string, INITIAL SAMPLE SIZE: string, REPLICATION SAMPLE SIZE: string, PLATFORM [SNPS PASSING QC]: string, ASSOCIATION COUNT: string, MAPPED_TRAIT: string, MAPPED_TRAIT_URI: string, STUDY ACCESSION: string, GENOTYPING TECHNOLOGY: string, SUBMISSION DATE: string, STATISTICAL MODEL: string, BACKGROUND TRAIT: string, MAPPED BACKGROUND TRAIT: string, MAPPED BACKGROUND TRAIT URI: string, COHORT: string, FULL SUMMARY STATISTICS: string, SUMMARY STATS LOCATION: string]

In [6]:
# Core Study Index Generation:
study_index = StudyIndexGWASCatalogParser.from_source(
    session.spark.read.csv(list(catalog_study_files), sep="\t", header=True),
    session.spark.read.csv(list(catalog_ancestry_files), sep="\t", header=True),
)

In [7]:
study_index.df.show()

+------------+---------+---------+--------+----------------------+---------------+--------------------+--------------------+--------------------+--------------------+------------------------+----------------------------------+--------------------+--------------------+---------------------+--------------------+------+---------+--------+
|     studyId|projectId|studyType|pubmedId|publicationFirstAuthor|publicationDate|  publicationJournal|    publicationTitle|     traitFromSource|   initialSampleSize|traitFromSourceMappedIds|backgroundTraitFromSourceMappedIds|             cohorts|    discoverySamples|ldPopulationStructure|  replicationSamples|nCases|nControls|nSamples|
+------------+---------+---------+--------+----------------------+---------------+--------------------+--------------------+--------------------+--------------------+------------------------+----------------------------------+--------------------+--------------------+---------------------+--------------------+------+------

In [9]:
gwas_catalog_study_curation = StudyIndexGWASCatalogOTCuration.from_csv(
            session, gwas_catalog_study_curation_file
        )

In [10]:
gwas_catalog_study_curation.show(2)

+----------+---------+-------------+---------------+---------+
|   studyId|studyType|analysisFlags|qualityControls|isCurated|
+----------+---------+-------------+---------------+---------+
|GCST000028|     null|           []|             []|     true|
|GCST000392|     null|           []|             []|     true|
+----------+---------+-------------+---------------+---------+
only showing top 2 rows



In [11]:
# Annotate with curation if provided:
if gwas_catalog_study_curation_file:
    if (gwas_catalog_study_curation_file.endswith(".tsv") | gwas_catalog_study_curation_file.endswith(".tsv")):
        gwas_catalog_study_curation = StudyIndexGWASCatalogOTCuration.from_csv(
            session, gwas_catalog_study_curation_file
        )
    elif gwas_catalog_study_curation_file.startswith("http"):
        gwas_catalog_study_curation = StudyIndexGWASCatalogOTCuration.from_url(
            session, gwas_catalog_study_curation_file,
        )
    else:
        raise ValueError(
            "Only CSV files or URLs are accepted as curation file."
        )
    study_index = study_index.annotate_from_study_curation(
        gwas_catalog_study_curation
    )

In [12]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType

# Annotate with sumstats QC if provided:
if sumstats_qc_path:
    schema = StructType([
        StructField("studyId", StringType(), True),
        StructField("mean_beta", DoubleType(), True),
        StructField("mean_diff_pz", DoubleType(), True),
        StructField("se_diff_pz", DoubleType(), True),
        StructField("gc_lambda", DoubleType(), True),
        StructField("n_variants", LongType(), True),
        StructField("n_variants_sig", LongType(), True)
    ])
    sumstats_qc = session.spark.read.schema(schema).parquet(sumstats_qc_path,recursiveFileLookup=True)

In [13]:
sumstats_qc.show(2)

+------------+--------------------+--------------------+--------------------+------------------+----------+--------------+
|     studyId|           mean_beta|        mean_diff_pz|          se_diff_pz|         gc_lambda|n_variants|n_variants_sig|
+------------+--------------------+--------------------+--------------------+------------------+----------+--------------+
|GCST90000047|-4.13776584488885...|0.001489360466501...| 0.06464638468525331| 1.356996071104317|  16298931|         17113|
|GCST90000448|4.499510274945693E-4|-1.67316909891496...|6.299456349660006E-5|1.0123659198314245|   7887487|             0|
+------------+--------------------+--------------------+--------------------+------------------+----------+--------------+
only showing top 2 rows



In [42]:
from gentropy.common.spark_helpers import convert_from_wide_to_long


cols = [c for c in sumstats_qc.columns if c != "studyId"]


melted_df = convert_from_wide_to_long(sumstats_qc, id_vars = ["studyId"], value_vars= cols, var_name="QCCheckName", value_name="QCCheckValue")
qc_df = melted_df.groupBy("studyId").agg(f.collect_list(f.struct(f.col("QCCheckName"), f.col("QCCheckValue"))).alias("sumStatQCValues")).withColumn("sumStatQCPerformed", f.lit(True))
qc_df.show(truncate=False)

+----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|studyId   |sumStatQCValues                                                                                                                                                    |sumStatQCPerformed|
+----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|GCST000569|[{mean_beta, -1.863105E-4}, {mean_diff_pz, -2.662145E-4}, {se_diff_pz, 0.015551462}, {gc_lambda, 1.0218483}, {n_variants, 2392760.0}, {n_variants_sig, 2.0}]       |true              |
|GCST000758|[{mean_beta, 1.5269937E-4}, {mean_diff_pz, 0.0034558668}, {se_diff_pz, 0.15163296}, {gc_lambda, 1.0103346}, {n_variants, 2626961.0}, {n_variants_sig, 1562.0}]     |true              |
|GCST000998|[{mean_b

In [39]:
import pyspark.sql.types as t
sumstats_qc_aggregated = sumstats_qc.groupBy("studyId").agg(
    f.collect_list(
        f.struct(
            *[
                f.struct(
                    f.lit(col).alias("QCCheckName"),
                    f.col(col).alias("QCCheckValue").cast(t.FloatType()),
                )
                for col in sumstats_qc.columns
                if col != "studyId"
            ]
        )
    ).alias("sumStatQCValues")
).withColumn("sumStatQCPerformed", f.lit(True))

In [37]:
sumstats_qc_aggregated.printSchema()

root
 |-- studyId: string (nullable = true)
 |-- sumStatQCValues: array (nullable = false)
 |    |-- element: struct (containsNull = false)
 |    |    |-- col1: struct (nullable = false)
 |    |    |    |-- QCCheckName: string (nullable = false)
 |    |    |    |-- col2: float (nullable = true)
 |    |    |-- col2: struct (nullable = false)
 |    |    |    |-- QCCheckName: string (nullable = false)
 |    |    |    |-- col2: float (nullable = true)
 |    |    |-- col3: struct (nullable = false)
 |    |    |    |-- QCCheckName: string (nullable = false)
 |    |    |    |-- col2: float (nullable = true)
 |    |    |-- col4: struct (nullable = false)
 |    |    |    |-- QCCheckName: string (nullable = false)
 |    |    |    |-- col2: float (nullable = true)
 |    |    |-- col5: struct (nullable = false)
 |    |    |    |-- QCCheckName: string (nullable = false)
 |    |    |    |-- col2: float (nullable = true)
 |    |    |-- col6: struct (nullable = false)
 |    |    |    |-- QCCheckName: 

In [31]:
sumstats_qc_aggregated.show(1,truncate=False)

+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|studyId   |sumStatQCValues                                                                                                                                               |sumStatQCPerformed|
+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
|GCST000569|[{{mean_beta, -1.863105E-4}, {mean_diff_pz, -2.662145E-4}, {se_diff_pz, 0.015551462}, {gc_lambda, 1.0218483}, {n_variants, 2392760.0}, {n_variants_sig, 2.0}}]|true              |
+----------+--------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+
only showing top 1 row



In [15]:
sumstats_qc_aggregated.count()

69286

In [43]:
df=study_index.df.drop("sumStatQCValues", "sumStatQCPerformed").join(qc_df, how="left", on="studyId").withColumn(
    "sumStatQCPerformed",
    f.coalesce(f.col("sumStatQCPerformed"), f.lit(False)),
).withColumn(
    "sumStatQCValues", f.coalesce(f.col("sumStatQCValues"), f.array())
)

In [44]:
df.count()

114252

In [45]:
df.show(1)

+----------+-------------------+--------------------+---------------+------------------+--------------------+----------------------------------+--------+---------+------------------+------------------------+----------------------+---------------------+-------------+-------+--------+--------------------+---------+---------+---------------+------+--------------------+------------------+
|   studyId|   discoverySamples|   initialSampleSize|publicationDate|publicationJournal|    publicationTitle|backgroundTraitFromSourceMappedIds|nSamples|studyType|replicationSamples|traitFromSourceMappedIds|publicationFirstAuthor|ldPopulationStructure|analysisFlags|cohorts|pubmedId|     traitFromSource|projectId|nControls|qualityControls|nCases|     sumStatQCValues|sumStatQCPerformed|
+----------+-------------------+--------------------+---------------+------------------+--------------------+----------------------------------+--------+---------+------------------+------------------------+-----------------

In [28]:
df.printSchema()

root
 |-- studyId: string (nullable = false)
 |-- discoverySamples: array (nullable = true)
 |    |-- element: struct (containsNull = false)
 |    |    |-- ancestry: string (nullable = true)
 |    |    |-- sampleSize: integer (nullable = false)
 |-- initialSampleSize: string (nullable = true)
 |-- publicationDate: string (nullable = true)
 |-- publicationJournal: string (nullable = true)
 |-- publicationTitle: string (nullable = true)
 |-- backgroundTraitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- nSamples: integer (nullable = true)
 |-- studyType: string (nullable = false)
 |-- replicationSamples: array (nullable = true)
 |    |-- element: struct (containsNull = false)
 |    |    |-- ancestry: string (nullable = true)
 |    |    |-- sampleSize: integer (nullable = true)
 |-- traitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- publicationFirstAuthor: string (nullable = true)
 |-- ld

In [46]:
StudyIndex(_df=df,_schema=StudyIndex.get_schema())

StudyIndex(_df=DataFrame[studyId: string, discoverySamples: array<struct<ancestry:string,sampleSize:int>>, initialSampleSize: string, publicationDate: string, publicationJournal: string, publicationTitle: string, backgroundTraitFromSourceMappedIds: array<string>, nSamples: int, studyType: string, replicationSamples: array<struct<ancestry:string,sampleSize:int>>, traitFromSourceMappedIds: array<string>, publicationFirstAuthor: string, ldPopulationStructure: array<struct<ldPopulation:string,relativeSampleSize:double>>, analysisFlags: array<string>, cohorts: array<string>, pubmedId: string, traitFromSource: string, projectId: string, nControls: int, qualityControls: array<string>, nCases: int, sumStatQCValues: array<struct<QCCheckName:string,QCCheckValue:float>>, sumStatQCPerformed: boolean], _schema=StructType([StructField('studyId', StringType(), False), StructField('projectId', StringType(), False), StructField('studyType', StringType(), False), StructField('traitFromSource', StringTyp

In [ ]:
from gentropy.sumstat_qc_step import SummaryStatisticsQCStep 
x=SummaryStatisticsQCStep(session=session,
gwas_path="gs://ot_orchestration/ukb_ppp_eur_data/harmonised_summary_statistics/",
output_path="gs://genetics-portal-dev-analysis/yt4/UKBB_PPP_QC_LOG_20241004",
pval_threshold= 1e-8)

In [3]:
x=SummaryStatistics.from_parquet(session,"gs://gwas_catalog_data/harmonised_summary_statistics/GCST003401.parquet")

In [4]:
x.df.count()

2119618

In [8]:
x.df=x.df.filter(f.col("pValueExponent")<=(-10000))

In [9]:
x.df.count()

0

In [10]:
from gentropy.method.sumstat_quality_controls import SummaryStatisticsQC

In [11]:
y=SummaryStatisticsQC.number_of_snps(x, pval_threshold=1e-8)

In [12]:
y.show()

+-------+----------+--------------+
|studyId|n_variants|n_variants_sig|
+-------+----------+--------------+
+-------+----------+--------------+



In [14]:
y=SummaryStatisticsQC.get_quality_control_metrics(gwas=x, pval_threshold=1e-8)

In [16]:
y.printSchema()

root
 |-- studyId: string (nullable = true)
 |-- mean_beta: double (nullable = true)
 |-- mean_diff_pz: double (nullable = true)
 |-- se_diff_pz: double (nullable = true)
 |-- gc_lambda: double (nullable = true)
 |-- n_variants: long (nullable = true)
 |-- n_variants_sig: long (nullable = true)



In [3]:
from gentropy.locus_breaker_clumping import LocusBreakerClumpingStep


ss1=LocusBreakerClumpingStep(session,
summary_statistics_input_path="gs://gwas_catalog_data/harmonised_summary_statistics/GCST003401.parquet",
clumped_study_locus_output_path="gs://genetics-portal-dev-analysis/yt4/tmp_v3",
lbc_baseline_pvalue=1e-5,
lbc_distance_cutoff=500_000,
lbc_pvalue_threshold=1.7e-7,
lbc_flanking_distance=250_000,
large_loci_size=1e6,
wbc_clump_distance=500_000,
wbc_pvalue_threshold=1.7e-7,
collect_locus = False,
remove_mhc = True)

In [7]:
type(1e6//2)

float

In [4]:
x.df.count()

2119618

In [5]:
from gentropy.sumstat_qc_step import SummaryStatisticsQCStep 

In [14]:
x=SummaryStatisticsQCStep(session=session,
                          gwas_path="gs://gwas_catalog_data/harmonised_summary_statistics/GCST003401.parquet",
        output_path="gs://genetics-portal-dev-analysis/yt4/sumstat_qc_output_log_example",
        pval_threshold= 1e-8)

In [12]:
y=session.spark.read.parquet("file:///Users/yt4/Projects/gentropy/notebooks/temp_files/sumstat_qc_output_log")

In [13]:
y.show(truncate=False)

+----------+----------------------+--------------------+-------------------+-----------------+----------+--------------+
|studyId   |mean_beta             |mean_diff_pz        |se_diff_pz         |gc_lambda        |n_variants|n_variants_sig|
+----------+----------------------+--------------------+-------------------+-----------------+----------+--------------+
|GCST003401|-1.0276427167536857E-5|-0.01287489120315704|0.04625858339667292|1.272534943024209|2119618   |1399          |
+----------+----------------------+--------------------+-------------------+-----------------+----------+--------------+



In [3]:
x=StudyLocus.from_parquet(session,"gs://ot_orchestration/ukb_ppp_eur_data/study_locus_datasets/locus_breaker_clumping")

In [5]:
x.df.count()

17670

In [6]:
x.df.show(2)

+---------+---------------+----------+--------+------+--------------------+---------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+------------+-----------+----------+--------+----------+-----+--------------------+----------+--------------------+
|studyType|      variantId|chromosome|position|region|             studyId|     beta|zScore|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|qualityControls|finemappingMethod|credibleSetIndex|credibleSetlog10BF|purityMeanR2|purityMinR2|locusStart|locusEnd|sampleSize|ldSet|               locus|confidence|        studyLocusId|
+---------+---------------+----------+--------+------+--------------------+---------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+-------------

In [11]:
y=session.spark.read.parquet("gs://genetics-portal-dev-analysis/dc16/output/ukb_ppp/clean_loci.parquet")
from pyspark.sql.types import StringType
# Assuming x is your DataFrame
y = y.withColumn("studyLocusId", f.col("studyLocusId").cast(StringType()))

df2=StudyLocus(
            _df=y,
            _schema=StudyLocus.get_schema(),
        )

AnalysisException: Path does not exist: gs://genetics-portal-dev-analysis/dc16/output/ukb_ppp/clean_loci.parquet

In [9]:
df2.df.count()

16015

In [10]:
df2.df.show(2)

+--------------------+---------------+----------+---------+------+--------------------+---------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+------------+-----------+----------+-----+--------------------+
|        studyLocusId|      variantId|chromosome| position|region|             studyId|     beta|zScore|pValueMantissa|pValueExponent|effectAlleleFrequencyFromSource|standardError|subStudyDescription|qualityControls|finemappingMethod|credibleSetIndex|credibleSetlog10BF|purityMeanR2|purityMinR2|sampleSize|ldSet|               locus|
+--------------------+---------------+----------+---------+------+--------------------+---------+------+--------------+--------------+-------------------------------+-------------+-------------------+---------------+-----------------+----------------+------------------+------------+-----------+----------+-----+--------------------

In [3]:
x=session.spark.read.parquet("temp_fiels/nan_locus2")
from pyspark.sql.types import StringType
# Assuming x is your DataFrame
x = x.withColumn("studyLocusId", f.col("studyLocusId").cast(StringType()))

df2=StudyLocus(
            _df=x,
            _schema=StudyLocus.get_schema(),
        )

df2=StudyLocus.from_parquet(session, "temp_fiels/nan_locus3")
df2=df2.df
#df2.show()

In [4]:
si=StudyIndex.from_parquet(session,"gs://genetics_etl_python_playground/releases/24.06/study_index/gwas_catalog")

In [5]:
session=session
study_locus_row=df2.collect()[1]
study_index=si
max_causal_snps=10
cs_lbf_thr=2
sum_pips=0.99
susie_est_tausq=False
run_carma=False
carma_tau=0.04
run_sumstat_imputation=False
carma_time_limit=6000
imputed_r2_threshold=0.8
ld_score_threshold=5

purity_mean_r2_threshold=0
purity_min_r2_threshold=0.25
lead_pval_threshold=1e-5
ld_min_r2=0.8

In [16]:
result_logging = SusieFineMapperStep.susie_finemapper_one_sl_row_v4_ss_gathered_boundaries(
    session=session,
    study_locus_row=study_locus_row,
    study_index=study_index,
    max_causal_snps=max_causal_snps,
    purity_mean_r2_threshold=purity_mean_r2_threshold,
    purity_min_r2_threshold=purity_min_r2_threshold,
    cs_lbf_thr=cs_lbf_thr,
    sum_pips=sum_pips,
    lead_pval_threshold=1e-5,
    susie_est_tausq=susie_est_tausq,
    run_carma=False,
    run_sumstat_imputation=run_sumstat_imputation,
    carma_tau=0.2,
    carma_time_limit=carma_time_limit,
    imputed_r2_threshold=imputed_r2_threshold,
    ld_score_threshold=ld_score_threshold,
    ld_min_r2=ld_min_r2,
)

In [17]:
result_logging["study_locus"] is not None

True

In [18]:
result_logging is not None

True

In [19]:
result_logging["log"]

,N_gwas_before_dedupl,N_gwas,N_ld,N_overlap,N_outliers,N_imputed,N_final_to_fm,elapsed_time
0,3498,3498,3498,3498,0,0,3498,25.492343


In [20]:
result_logging["study_locus"].df.show(truncate=False)

+----------------+--------------------------------+------------+--------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------+---------+-----------------+------------------+------------+-----------+-------------------+--------------+--------------+----------+---------+
|credibleSetIndex|studyLocusId                    |studyId     |region              |locus                                                                                                                                                                                                                                                                                                                            |variantId     

In [21]:
variant_id_list = result_logging["study_locus"].df.select("variantId").rdd.flatMap(lambda x: x).collect()
variant_id_to_index = {variant: idx for idx, variant in enumerate(result_logging["GWAS_df"]['variantId'])}
indices = [variant_id_to_index[variant] for variant in variant_id_list if variant in variant_id_to_index]

In [22]:
variant_id_list

['7_100638579_G_A',
 '7_100621008_C_T',
 '7_100717534_C_CT',
 '7_99613384_A_G',
 '7_99970285_C_T',
 '7_99922677_C_CA',
 '7_99969408_T_TA']

In [23]:
indices

[3207, 3162, 3431, 922, 1742, 1586, 1740]

In [24]:
result_logging["GWAS_df"].iloc[indices]

,index,variantId,z
3207,3207,7_100638579_G_A,22.032926
3162,3162,7_100621008_C_T,11.436719
3431,3431,7_100717534_C_CT,-10.770291
922,922,7_99613384_A_G,9.024065
1742,1742,7_99970285_C_T,-7.650038
1586,1586,7_99922677_C_CA,-5.191238
1740,1740,7_99969408_T_TA,5.180491


In [25]:
x=result_logging["LD"][indices, :][:, indices]
x=x**2
x[x>=ld_min_r2]=1
x[x<ld_min_r2]=0
x

array([[1., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 1.]])

In [6]:
import logging
import time
from typing import Any

import numpy as np
import pandas as pd
import pyspark.sql.functions as f
import scipy as sc
from pyspark.sql import DataFrame, Row, Window
from pyspark.sql.functions import row_number
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

from gentropy.common.session import Session
from gentropy.common.spark_helpers import (
    neglog_pvalue_to_mantissa_and_exponent,
    order_array_of_structs_by_field,
)
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.datasource.gnomad.ld import GnomADLDMatrix
from gentropy.method.carma import CARMA
from gentropy.method.sumstat_imputation import SummaryStatisticsImputation
from gentropy.method.susie_inf import SUSIE_inf

In [7]:
# PLEASE DO NOT REMOVE THIS LINE
pd.DataFrame.iteritems = pd.DataFrame.items

chromosome = study_locus_row["chromosome"]
studyId = study_locus_row["studyId"]
locusStart = study_locus_row["locusStart"]
locusEnd = study_locus_row["locusEnd"]

study_index_df = study_index._df
study_index_df = study_index_df.filter(f.col("studyId") == studyId)
major_population = study_index_df.select(
    "studyId",
    order_array_of_structs_by_field(
        "ldPopulationStructure", "relativeSampleSize"
    )[0]["ldPopulation"].alias("majorPopulation"),
).collect()[0]["majorPopulation"]

region = chromosome + ":" + str(int(locusStart)) + "-" + str(int(locusEnd))

schema = StudyLocus.get_schema()
gwas_df = session.spark.createDataFrame([study_locus_row], schema=schema)
exploded_df = gwas_df.select(f.explode("locus").alias("locus"))

result_df = exploded_df.select(
    "locus.variantId", "locus.beta", "locus.standardError"
)
gwas_df = (
    result_df.withColumn("z", f.col("beta") / f.col("standardError"))
    .withColumn(
        "chromosome", f.split(f.col("variantId"), "_")[0].cast("string")
    )
    .withColumn("position", f.split(f.col("variantId"), "_")[1].cast("int"))
    .filter(f.col("chromosome") == chromosome)
    .filter(f.col("position") >= int(locusStart))
    .filter(f.col("position") <= int(locusEnd))
    .filter(f.col("z").isNotNull())
)

# Remove ALL duplicated variants from GWAS DataFrame - we don't know which is correct
variant_counts = gwas_df.groupBy("variantId").count()
unique_variants = variant_counts.filter(f.col("count") == 1)
gwas_df = gwas_df.join(unique_variants, on="variantId", how="left_semi")

ld_index = (
    GnomADLDMatrix()
    .get_locus_index_boundaries(
        study_locus_row=study_locus_row,
        major_population=major_population,
    )
    .withColumn(
        "variantId",
        f.concat(
            f.lit(chromosome),
            f.lit("_"),
            f.col("`locus.position`"),
            f.lit("_"),
            f.col("alleles").getItem(0),
            f.lit("_"),
            f.col("alleles").getItem(1),
        ).cast("string"),
    )
)
# Remove ALL duplicated variants from ld_index DataFrame - we don't know which is correct
variant_counts = ld_index.groupBy("variantId").count()
unique_variants = variant_counts.filter(f.col("count") == 1)
ld_index = ld_index.join(unique_variants, on="variantId", how="left_semi").sort(
    "idx"
)

if not run_sumstat_imputation:
    # Filtering out the variants that are not in the LD matrix, we don't need them
    gwas_index = gwas_df.join(
        ld_index.select("variantId", "alleles", "idx"), on="variantId"
    ).sort("idx")
    gwas_df = gwas_index.select(
        "variantId",
        "z",
        "chromosome",
        "position",
        "beta",
        "StandardError",
    )
    gwas_index = gwas_index.drop(
        "z", "chromosome", "position", "beta", "StandardError"
    )

    gnomad_ld = GnomADLDMatrix.get_numpy_matrix(
        gwas_index, gnomad_ancestry=major_population
    )

    # Module to remove NANs from the LD matrix
    if sum(sum(np.isnan(gnomad_ld))) > 0:
        gwas_index = gwas_index.toPandas()

        # First round of filtering out the variants with NANs
        nan_count = 1 - (sum(np.isnan(gnomad_ld)) / len(gnomad_ld))
        indices = np.where(nan_count >= 0.98)
        indices = indices[0]
        gnomad_ld = gnomad_ld[indices][:, indices]

        gwas_index = gwas_index.iloc[indices, :]


        # Second round of filtering out the variants with NANs
        nan_count = sum(np.isnan(gnomad_ld))
        indices = np.where(nan_count == 0)
        indices = indices[0]

        gnomad_ld = gnomad_ld[indices][:, indices]
        gwas_index = gwas_index.iloc[indices, :]

 
        gwas_index = session.spark.createDataFrame(gwas_index)

else:
    gwas_index = gwas_df.join(
        ld_index.select("variantId", "alleles", "idx"), on="variantId"
    ).sort("idx")

    gwas_index = ld_index
    gnomad_ld = GnomADLDMatrix.get_numpy_matrix(
        gwas_index, gnomad_ancestry=major_population
    )

    # Module to remove NANs from the LD matrix
    if sum(sum(np.isnan(gnomad_ld))) > 0:
        gwas_index = gwas_index.toPandas()

        # First round of filtering out the variants with NANs
        nan_count = 1 - (sum(np.isnan(gnomad_ld)) / len(gnomad_ld))
        indices = np.where(nan_count >= 0.98)
        indices = indices[0]
        gnomad_ld = gnomad_ld[indices][:, indices]

        gwas_index = gwas_index.iloc[indices, :]



        # Second round of filtering out the variants with NANs
        nan_count = sum(np.isnan(gnomad_ld))
        indices = np.where(nan_count == 0)
        indices = indices[0]

        gnomad_ld = gnomad_ld[indices][:, indices]
        gwas_index = gwas_index.iloc[indices, :]


        gwas_index = session.spark.createDataFrame(gwas_index)

# sanity filters on LD matrix
np.fill_diagonal(gnomad_ld, 1)
gnomad_ld[gnomad_ld > 1] = 1
gnomad_ld[gnomad_ld < -1] = -1
upper_triangle = np.triu(gnomad_ld)
gnomad_ld = (
    upper_triangle + upper_triangle.T - np.diag(upper_triangle.diagonal())
)
np.fill_diagonal(gnomad_ld, 1)

ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/Users/yt4/Library/Caches/pypoetry/virtualenvs/gentropy-krNFZEZg-py3.10/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/yt4/Library/Caches/pypoetry/virtualenvs/gentropy-krNFZEZg-py3.10/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/Users/yt4/.pyenv/versions/3.10.8/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


2024-10-01 15:15:35.210 Hail: INFO: Table.join: renamed the following fields on the right to avoid name conflicts:
    'age_distribution' -> 'age_distribution_1'
    'freq_index_dict' -> 'freq_index_dict_1'
    'popmax_index_dict' -> 'popmax_index_dict_1'
    'rf' -> 'rf_1'
    'faf_index_dict' -> 'faf_index_dict_1'
    'freq_meta' -> 'freq_meta_1'
    'age_index_dict' -> 'age_index_dict_1'


KeyboardInterrupt: 

In [20]:
gnomad_ld.shape

(3087, 3087)

In [10]:
gwas_index.count()

3087

In [11]:
GWAS_df=gwas_df
ld_index=gwas_index
gnomad_ld=gnomad_ld
L=max_causal_snps
session=session
studyId=studyId
region=region
susie_est_tausq=susie_est_tausq
run_carma=run_carma
run_sumstat_imputation=run_sumstat_imputation
carma_time_limit=carma_time_limit
carma_tau=carma_tau
imputed_r2_threshold=imputed_r2_threshold
ld_score_threshold=ld_score_threshold
sum_pips=sum_pips

In [12]:
# PLEASE DO NOT REMOVE THIS LINE
pd.DataFrame.iteritems = pd.DataFrame.items

start_time = time.time()
GWAS_df = GWAS_df.toPandas()
N_gwas_before_dedupl = len(GWAS_df)

GWAS_df = GWAS_df.drop_duplicates(subset="variantId", keep=False)
GWAS_df = GWAS_df.reset_index()

ld_index = ld_index.toPandas()
ld_index = ld_index.reset_index()

N_gwas = len(GWAS_df)
N_ld = len(ld_index)

# Filtering out the variants that are not in the LD matrix, we don't need them
df_columns = ["variantId", "z"]
GWAS_df = GWAS_df.merge(ld_index, on="variantId", how="inner")
GWAS_df = GWAS_df[df_columns].reset_index()
N_after_merge = len(GWAS_df)

merged_df = GWAS_df.merge(
    ld_index, left_on="variantId", right_on="variantId", how="inner"
)
indices = merged_df["index_y"].values

ld_to_fm = gnomad_ld[indices][:, indices]
z_to_fm = GWAS_df["z"].values

if run_carma:
    carma_output = CARMA.time_limited_CARMA_spike_slab_noEM(
        z=z_to_fm, ld=ld_to_fm, sec_threshold=carma_time_limit, tau=carma_tau
    )
    if carma_output["Outliers"] != [] and carma_output["Outliers"] is not None:
        GWAS_df.drop(carma_output["Outliers"], inplace=True)
        GWAS_df = GWAS_df.reset_index()
        ld_index = ld_index.reset_index()
        merged_df = GWAS_df.merge(
            ld_index, left_on="variantId", right_on="variantId", how="inner"
        )
        indices = merged_df["index_y"].values

        ld_to_fm = gnomad_ld[indices][:, indices]
        z_to_fm = GWAS_df["z"].values
        N_outliers = len(carma_output["Outliers"])
    else:
        N_outliers = 0
else:
    N_outliers = 0

if run_sumstat_imputation:
    known = indices
    unknown = [
        index for index in list(range(len(gnomad_ld))) if index not in known
    ]
    sig_t = gnomad_ld[known, :][:, known]
    sig_i_t = gnomad_ld[unknown, :][:, known]
    zt = z_to_fm

    sumstat_imp_res = SummaryStatisticsImputation.raiss_model(
        z_scores_known=zt,
        ld_matrix_known=sig_t,
        ld_matrix_known_missing=sig_i_t,
        lamb=0.01,
        rtol=0.01,
    )

    bool_index = (sumstat_imp_res["imputation_r2"] >= imputed_r2_threshold) * (
        sumstat_imp_res["ld_score"] >= ld_score_threshold
    )
    if sum(bool_index) >= 1:
        indices = np.where(bool_index)[0]
        index_to_add = [unknown[i] for i in indices]
        index_to_fm = np.concatenate((known, index_to_add))

        ld_to_fm = gnomad_ld[index_to_fm][:, index_to_fm]

        snp_info_to_add = pd.DataFrame(
            {
                "variantId": ld_index.iloc[index_to_add, :]["variantId"],
                "z": sumstat_imp_res["mu"][indices],
            }
        )
        GWAS_df = pd.concat([GWAS_df, snp_info_to_add], ignore_index=True)
        z_to_fm = GWAS_df["z"].values

        N_imputed = len(indices)
    else:
        N_imputed = 0
else:
    N_imputed = 0

susie_output = SUSIE_inf.susie_inf(
    z=z_to_fm, LD=ld_to_fm, L=L, est_tausq=susie_est_tausq
)

schema = StructType(
    [
        StructField("variantId", StringType(), True),
        StructField("z", DoubleType(), True),
    ]
)
variant_index = (
    session.spark.createDataFrame(
        GWAS_df[["variantId", "z"]],
        schema=schema,
    )
    .withColumn(
        "chromosome", f.split(f.col("variantId"), "_")[0].cast("string")
    )
    .withColumn("position", f.split(f.col("variantId"), "_")[1].cast("int"))
)

In [13]:
susie_output

{'PIP': array([[1.01952713e-55, 3.77183367e-12, 0.00000000e+00, ...,
         9.92106496e-49, 2.65700035e-12, 2.32022272e-56],
        [1.71119348e-54, 5.64004050e-11, 0.00000000e+00, ...,
         1.83274434e-47, 4.49528792e-12, 3.10650968e-55],
        [1.03952330e-55, 3.86169306e-12, 0.00000000e+00, ...,
         1.02390098e-48, 2.50815896e-12, 2.38139879e-56],
        ...,
        [9.18199368e-56, 3.54896586e-12, 0.00000000e+00, ...,
         8.89186746e-49, 2.29414421e-12, 2.16036911e-56],
        [1.08252407e-05, 7.00170215e-11, 0.00000000e+00, ...,
         3.03519949e-24, 4.53366085e-11, 1.23030396e-40],
        [1.01759382e-55, 5.38264112e-12, 0.00000000e+00, ...,
         1.01846032e-48, 4.29630166e-12, 2.47062003e-56]]),
 'mu': array([[-5.43962982e-04, -4.55719606e-04, -7.51670254e-04, ...,
         -2.79725944e-04, -1.28516100e-03, -4.44932182e-04],
        [-6.59472240e-03, -6.42173013e-03, -8.03372366e-03, ...,
         -6.68762350e-03, -3.24227791e-03, -6.33049105e-03],


In [14]:
susie_output=susie_output
session=session
studyId=studyId
region=region
variant_index=variant_index
sum_pips=sum_pips
ld_matrix=ld_to_fm
cs_lbf_thr=cs_lbf_thr

In [15]:
# PLEASE DO NOT REMOVE THIS LINE
pd.DataFrame.iteritems = pd.DataFrame.items

variants = np.array(
    [row["variantId"] for row in variant_index.select("variantId").collect()]
).reshape(-1, 1)

PIPs = susie_output["PIP"]
lbfs = susie_output["lbf_variable"]
mu = susie_output["mu"]
susie_result = np.hstack((variants, PIPs, lbfs, mu))

L_snps = PIPs.shape[1]

# Extracting credible sets
order_creds = list(enumerate(susie_output["lbf"]))
order_creds.sort(key=lambda x: x[1], reverse=True)

counter = 0
for i, cs_lbf_value in order_creds:
    if counter > 0 and cs_lbf_value < cs_lbf_thr:
        counter += 1
        continue
    counter += 1
    sorted_arr = susie_result[
        susie_result[:, i + 1].astype(float).argsort()[::-1]
    ]
    cumsum_arr = np.cumsum(sorted_arr[:, i + 1].astype(float))
    filter_row = np.argmax(cumsum_arr >= sum_pips)
    if filter_row == 0 and cumsum_arr[0] < sum_pips:
        filter_row = len(cumsum_arr)
    filter_row += 1
    filtered_arr = sorted_arr[:filter_row]
    cred_set = filtered_arr[:, [0, i + 1, i + L_snps + 1, i + 2 * L_snps + 1]]
    win = Window.rowsBetween(
        Window.unboundedPreceding, Window.unboundedFollowing
    )

    cred_set = (
        session.spark.createDataFrame(
            cred_set.tolist(),
            ["variantId", "posteriorProbability", "logBF", "beta"],
        )
        .join(
            variant_index.select(
                "variantId",
                "chromosome",
                "position",
            ),
            "variantId",
        )
        .sort(f.desc("posteriorProbability"))
        .withColumn(
            "locus",
            f.collect_list(
                f.struct(
                    f.col("variantId").cast("string").alias("variantId"),
                    f.col("posteriorProbability")
                    .cast("double")
                    .alias("posteriorProbability"),
                    f.col("logBF").cast("double").alias("logBF"),
                    f.col("beta").cast("double").alias("beta"),
                )
            ).over(win),
        )
        .limit(1)
        .withColumns(
            {
                "studyId": f.lit(studyId),
                "region": f.lit(region),
                "credibleSetIndex": f.lit(counter),
                "credibleSetlog10BF": f.lit(cs_lbf_value * 0.4342944819),
                "finemappingMethod": f.lit("SuSiE-inf"),
            }
        )
        .withColumn(
            "studyLocusId",
            StudyLocus.assign_study_locus_id(
                f.col("studyId"), f.col("variantId"), f.col("finemappingMethod")
            ),
        )
        .select(
            "studyLocusId",
            "studyId",
            "region",
            "credibleSetIndex",
            "locus",
            "variantId",
            "chromosome",
            "position",
            "finemappingMethod",
            "credibleSetlog10BF",
        )
    )
    if counter == 1:
        cred_sets = cred_set
    else:
        cred_sets = cred_sets.unionByName(cred_set)

# Calulating purity
variant_index_df = variant_index.toPandas()
cred_sets_variantId = cred_sets.select("locus.variantId").toPandas()

lead_variantId_list = (
    cred_sets.select("variantId").toPandas()["variantId"].tolist()
)
cred_set_index = (
    cred_sets.select("credibleSetIndex").toPandas()["credibleSetIndex"].tolist()
)
vlist_series = pd.Series(lead_variantId_list)
ind = vlist_series.map(variant_index_df.set_index("variantId").index.get_loc)
z_values = variant_index_df.iloc[ind]["z"].tolist()
z_values_array = np.array(z_values)
pval = sc.stats.chi2.sf((z_values_array**2), 1)

# sometimes pval is 0, we need to avoid it
pval[pval < 1e-322] = 1e-322

neglogpval = -np.log10(pval)
neglogpval = neglogpval.tolist()

list_purity_mean_r2 = []
list_purity_min_r2 = []
for _, row in cred_sets_variantId.iterrows():
    row = row.iloc[0]
    vlist_series = pd.Series(row)
    ind = vlist_series.map(
        variant_index_df.set_index("variantId").index.get_loc
    )
    # print(variant_index_df.iloc[ind,0]==vlist)
    squared_matrix = ld_matrix[ind, :][:, ind] ** 2
    purity_mean_r2 = np.mean(squared_matrix)
    purity_min_r2 = np.min(squared_matrix)
    list_purity_mean_r2.append(purity_mean_r2)
    list_purity_min_r2.append(purity_min_r2)

cred_sets = cred_sets.drop("pValueMantissa", "pValueExponent")

df = pd.DataFrame(
    {
        "credibleSetIndex": cred_set_index,
        "purityMeanR2": purity_mean_r2,
        "purityMinR2": purity_min_r2,
        "zScore": z_values,
        "neglogpval": neglogpval,
    }
)
schema = StructType(
    [
        StructField("credibleSetIndex", IntegerType(), True),
        StructField("purityMeanR2", DoubleType(), True),
        StructField("purityMinR2", DoubleType(), True),
        StructField("zScore", DoubleType(), True),
        StructField("neglogpval", DoubleType(), True),
    ]
)

df_spark = session.spark.createDataFrame(df, schema=schema)

cred_sets = cred_sets.join(df_spark, on="credibleSetIndex")

mantissa, exponent = neglog_pvalue_to_mantissa_and_exponent(
    cred_sets.neglogpval
)
cred_sets = cred_sets.withColumn("pValueMantissa", mantissa)
cred_sets = cred_sets.withColumn("pValueExponent", exponent)
cred_sets = cred_sets.withColumn(
    "pValueMantissa", f.col("pValueMantissa").cast("float")
)

In [16]:
cred_sets.show(truncate=False)

+----------------+--------------------+------------+--------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------+--------+-----------------+------------------+------------+-----------+-------------------+-------------------+--------------+--------------+
|credibleSetIndex|studyLocusId        |studyId     |region              |locus                                                                                                                                                                                                                                                |variantId      |chromosome|position|finemappingMethod|credibleSetlog10BF|purityMeanR2|purityMinR2|zScore             |neglogpval         |pValueMantissa|pValueExponent|
+----------------+------

In [20]:
lead_pval_threshold=1e-3
purity_mean_r2_threshold=0.01
purity_min_r2_threshold=0.01
cs_lbf_thr=2
ld_min_r2=0.8

In [61]:
from pyspark.sql.functions import desc, row_number
# Filter by lead p-value, credible set logBF, purity mean r2 and purity min r2
cred_sets = cred_sets.filter(
    (f.col("neglogpval") >= -np.log10(lead_pval_threshold))
    & (f.col("credibleSetlog10BF") >= cs_lbf_thr * 0.4342944819)
    & (f.col("purityMinR2") >= purity_min_r2_threshold)
    & (f.col("purityMeanR2") >= purity_mean_r2_threshold)
)


# Remove duplicated by lead variant
if cred_sets.count() > 1:
    window = Window.partitionBy("variantId").orderBy("credibleSetIndex")
    cred_sets = cred_sets.withColumn("rank", row_number().over(window))
    cred_sets = cred_sets.filter(cred_sets["rank"] == 1).drop("rank")
    cred_sets = cred_sets.orderBy("credibleSetIndex")

# Remove CSs with high LD between leads
if cred_sets.count() > 1:
    cred_sets = cred_sets.orderBy(desc("neglogpval"))
    lead_variantId_list = (
        cred_sets.select("variantId").toPandas()["variantId"].tolist()
    )
    vlist_series = pd.Series(lead_variantId_list)
    ind = vlist_series.map(
        variant_index_df.set_index("variantId").index.get_loc
    )
    ld_leads = ld_matrix[ind, :][:, ind]
    ld_leads = ld_leads**2
    ld_leads = ld_leads - np.tril(ld_leads)
    np.fill_diagonal(ld_leads, 0)


In [62]:
cred_sets.show(truncate=False)

+----------------+--------------------+------------+--------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------+--------+-----------------+------------------+------------+-----------+-------------------+------------------+--------------+--------------+
|credibleSetIndex|studyLocusId        |studyId     |region              |locus                                                                                                                                                                                                                                                |variantId      |chromosome|position|finemappingMethod|credibleSetlog10BF|purityMeanR2|purityMinR2|zScore             |neglogpval        |pValueMantissa|pValueExponent|
+----------------+--------

In [63]:
lead_variantId_list

['15_43658501_A_T',
 '15_43331199_T_G',
 '15_43481838_C_G',
 '15_43509840_A_C',
 '15_43520398_G_C',
 '15_43297635_A_G']

In [64]:
lead_variantId_list_to_delete = []
for idx in range(len(lead_variantId_list)):
    vId = lead_variantId_list[idx]
    if vId in lead_variantId_list_to_delete:
        continue
    high_ld_indices = np.where(ld_leads[idx, :] >= ld_min_r2)[0]
    if len(high_ld_indices) > 0:
        lead_variantId_list_to_delete=lead_variantId_list_to_delete+list(np.array(lead_variantId_list)[high_ld_indices])

In [65]:
lead_variantId_list_to_delete

['15_43509840_A_C']

In [67]:
ld_leads[ld_leads >= ld_min_r2] = 1
ld_leads[ld_leads < ld_min_r2] = 0

In [68]:
ld_leads

array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])

In [69]:
lead_variantId_list

['15_43658501_A_T',
 '15_43331199_T_G',
 '15_43481838_C_G',
 '15_43509840_A_C',
 '15_43520398_G_C',
 '15_43297635_A_G']

In [43]:
high_ld_indices = np.where(ld_leads[idx, :] >= ld_min_r2)[0]

In [55]:
list(np.array(lead_variantId_list)[high_ld_indices])

['15_43509840_A_C']

In [45]:
len(high_ld_indices)

1

In [42]:
lead_variantId_list_to_delete

[]

In [29]:
idx=0
vId = lead_variantId_list[idx]

In [33]:
high_ld_indices = np.where(ld_leads[idx, :] >= ld_min_r2)[0]

In [34]:
high_ld_indices

array([], dtype=int64)

In [37]:
high_ld_indices = np.where(ld_leads[idx, :] >= ld_min_r2)[0]

In [39]:
len(high_ld_indices)


0

In [22]:
cred_sets2 = cred_sets.filter(
    (f.col("neglogpval") >= -np.log10(lead_pval_threshold))
    & (f.col("credibleSetlog10BF") >= cs_lbf_thr * 0.4342944819)
    & (f.col("purityMinR2") >= purity_min_r2_threshold)
    & (f.col("purityMeanR2") >= purity_mean_r2_threshold)
)

In [23]:
cred_sets2.show(truncate=False)

+----------------+--------------------+------------+--------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------+--------+-----------------+------------------+------------+-----------+-------------------+------------------+--------------+--------------+
|credibleSetIndex|studyLocusId        |studyId     |region              |locus                                                                                                                                                                                                                                                |variantId      |chromosome|position|finemappingMethod|credibleSetlog10BF|purityMeanR2|purityMinR2|zScore             |neglogpval        |pValueMantissa|pValueExponent|
+----------------+--------

In [24]:
cred_sets2.count()

6

In [44]:
cred_sets=cred_sets2

In [45]:
window = Window.partitionBy("variantId").orderBy("credibleSetIndex")
cred_sets = cred_sets.withColumn("rank", row_number().over(window))
cred_sets = cred_sets.filter(cred_sets["rank"] == 1).drop("rank")
cred_sets = cred_sets.orderBy("credibleSetIndex")

In [46]:
cred_sets.count()

6

In [52]:
from pyspark.sql.functions import desc

cred_sets = cred_sets.orderBy(desc("neglogpval"))

In [53]:
lead_variantId_list = (
    cred_sets.select("variantId").toPandas()["variantId"].tolist()
)

In [54]:
lead_variantId_list

['15_43658501_A_T',
 '15_43331199_T_G',
 '15_43481838_C_G',
 '15_43509840_A_C',
 '15_43520398_G_C',
 '15_43297635_A_G']

In [55]:
cred_sets.show(truncate=False)

+----------------+--------------------+------------+--------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------+--------+-----------------+------------------+------------+-----------+-------------------+------------------+--------------+--------------+
|credibleSetIndex|studyLocusId        |studyId     |region              |locus                                                                                                                                                                                                                                                |variantId      |chromosome|position|finemappingMethod|credibleSetlog10BF|purityMeanR2|purityMinR2|zScore             |neglogpval        |pValueMantissa|pValueExponent|
+----------------+--------

In [56]:
vlist_series = pd.Series(lead_variantId_list)

In [57]:
vlist_series

0    15_43658501_A_T
1    15_43331199_T_G
2    15_43481838_C_G
3    15_43509840_A_C
4    15_43520398_G_C
5    15_43297635_A_G
dtype: object

In [58]:
ind = vlist_series.map(
    variant_index_df.set_index("variantId").index.get_loc
)

In [59]:
ind

0    2982
1    2279
2    2659
3    2728
4    2741
5    2207
dtype: int64

In [60]:
ld_leads = ld_matrix[ind, :][:, ind]
ld_leads = ld_leads**2

In [61]:
ld_leads

array([[1.00000000e+00, 4.23977780e-01, 2.02906542e-01, 1.97427250e-01,
        2.12597673e-01, 3.40106831e-04],
       [4.23977780e-01, 1.00000000e+00, 1.03494589e-01, 9.93705632e-02,
        1.05278816e-01, 3.33426953e-05],
       [2.02906542e-01, 1.03494589e-01, 1.00000000e+00, 8.17014822e-01,
        7.67409092e-01, 7.45504130e-03],
       [1.97427250e-01, 9.93705632e-02, 8.17014822e-01, 1.00000000e+00,
        8.58464755e-01, 1.32570702e-02],
       [2.12597673e-01, 1.05278816e-01, 7.67409092e-01, 8.58464755e-01,
        1.00000000e+00, 1.72855712e-02],
       [3.40106831e-04, 3.33426953e-05, 7.45504130e-03, 1.32570702e-02,
        1.72855712e-02, 1.00000000e+00]])

In [62]:
ld_min_r2=0.8

In [63]:
ld_leads[ld_leads >= ld_min_r2] = 1
ld_leads[ld_leads < ld_min_r2] = 0
np.fill_diagonal(ld_leads, 0)

In [64]:
ld_leads

array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 1., 0., 1., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])

In [65]:
ld_leads = ld_leads - np.tril(ld_leads)

In [66]:
ld_leads

array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])

In [67]:
column_sums = ld_leads.sum(axis=0)

In [68]:
column_sums

array([0., 0., 0., 1., 1., 0.])

In [69]:
sum(column_sums) > 0

True

In [87]:
non_zero_mask = column_sums != 0

# Apply the mask to lead_variantId_list
lead_variantId_list_to_delete = list(np.array(lead_variantId_list)[non_zero_mask])


In [88]:
list(lead_variantId_list_to_delete)

['15_43509840_A_C', '15_43520398_G_C']

In [86]:
ld_leads

array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 0., 0.],
       [0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])

In [89]:
for vId in lead_variantId_list_to_delete:
    cred_sets = cred_sets.filter(f.col("variantId") != vId)

In [93]:
cred_sets.show(truncate=False)

+----------------+--------------------+------------+--------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+----------+--------+-----------------+------------------+------------+-----------+-------------------+--------------+--------------+
|credibleSetIndex|studyLocusId        |studyId     |region              |locus                                                                                                                                                                                                                                                |variantId      |chromosome|position|finemappingMethod|credibleSetlog10BF|purityMeanR2|purityMinR2|zScore             |pValueMantissa|pValueExponent|
+----------------+--------------------+------------+------------

In [92]:
cred_sets = cred_sets.drop("neglogpval")

In [94]:
StudyLocus(
            _df=cred_sets,
            _schema=StudyLocus.get_schema(),
        )

StudyLocus(_df=DataFrame[credibleSetIndex: int, studyLocusId: bigint, studyId: string, region: string, locus: array<struct<variantId:string,posteriorProbability:double,logBF:double,beta:double>>, variantId: string, chromosome: string, position: int, finemappingMethod: string, credibleSetlog10BF: double, purityMeanR2: double, purityMinR2: double, zScore: double, pValueMantissa: float, pValueExponent: int], _schema=StructType([StructField('studyLocusId', LongType(), False), StructField('studyType', StringType(), True), StructField('variantId', StringType(), False), StructField('chromosome', StringType(), True), StructField('position', IntegerType(), True), StructField('region', StringType(), True), StructField('studyId', StringType(), False), StructField('beta', DoubleType(), True), StructField('zScore', DoubleType(), True), StructField('pValueMantissa', FloatType(), True), StructField('pValueExponent', IntegerType(), True), StructField('effectAlleleFrequencyFromSource', FloatType(), Tru